# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faith-amanze/assignment1/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [13]:
import pandas as pd
import numpy as np

df = pd.read_csv("/workspaces/assignment1/data/raw/content_refresh_anonymized.csv")
print(df.shape)
print(df.columns.tolist())
# ── Hygiene: drop rows with no position data before using avg_position ──
n_total = len(df)
df_pos = df[df["avg_position"] > 0].copy()
print(f"n total = {n_total}, n with valid avg_position = {len(df_pos)}")

# ── Signal 1: staleness (flag-linked — behind refresh flags) ──
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 30, 90, 180, 365, 99999],
    labels=["0-30", "31-90", "91-180", "181-365", "365+"]
)

signal1_table = df.groupby("staleness_bucket", observed=True).agg(
    n=("staleness_bucket", "size"),
    avg_ctr=("ctr", "mean"),
    avg_engagement_rate=("engagement_rate", "mean"),
).reset_index()
print("\nSignal 1 — Staleness vs CTR/engagement")
print(signal1_table)
# ── Signal 2: CTR vs position (flag-linked — behind CTR-fix logic) ──
df_pos["position_bucket"] = pd.cut(
    df_pos["avg_position"],
    bins=[0, 3, 10, 20, 50, 1000],
    labels=["1-3", "4-10", "11-20", "21-50", "50+"]
)

signal2_table = df_pos.groupby("position_bucket", observed=True).agg(
    n=("position_bucket", "size"),
    avg_ctr=("ctr", "mean"),
).reset_index()
print("\nSignal 2 — Position vs CTR")
print(signal2_table)



(30000, 44)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
n total = 30000, n with valid avg_position = 28795

Signal 1 — Staleness vs CTR/engagement
  staleness_bucket      n    avg_ctr  avg_engagement_rate
0             0-30  20480   0.609021             2.599727
1         

 # Signal 1 verdict: MIXED
# CTR does not move monotonically with staleness. The 0-30 and 91-180
# buckets (n=20,480 and n=9,171) show CTR of 0.61% and 0.24%, but the small
# buckets (31-90: n=175, 181-365: n=169, 365+: n=5) swing wildly and are too
# thin to trust — especially 365+ at n=5, which is noise, not signal.

# Signal 2 verdict: CONFIRMED
# CTR drops monotonically as avg_position worsens (2.71% at positions 1-3
# down to 0.15% at 50+), across buckets with n ranging from 1,141 to 11,842.
# Clean, reliable signal — safe to build the rule on.

In [17]:
# ── The rule, in plain words ──
# "A page is worth reviewing if it ranks in a weak position (11+) but still
#  gets meaningful search visibility (impressions) — it has an audience,
#  it's just underperforming on clicks because of where it ranks."

weak_position = (df_pos["avg_position"] >= 11).astype(int)
has_visibility = (df_pos["impressions_90d"] >= 500).astype(int)

df_pos["score"] = weak_position * has_visibility * df_pos["impressions_90d"]

df_pos["reason_code"] = np.where(
    (weak_position == 1) & (has_visibility == 1), "weak_position_visible",
    np.where(weak_position == 1, "weak_position_low_visibility", "strong_position")
)

df_pos["action"] = np.where(df_pos["score"] > 0, "REVIEW_CTR", "NO_ACTION")

print(df_pos[["content_id", "avg_position", "impressions_90d", "score", "reason_code", "action"]].sort_values("score", ascending=False).head(10))

                 content_id  avg_position  impressions_90d   score  \
19636  content_2cb567c3c89b          22.2           497727  497727   
29400  content_2dba2b1f9536          27.9           443434  443434   
26798  content_b28d1efd668f          26.2           286608  286608   
23767  content_813e88069237          26.2           233561  233561   
26304  content_ff94c9b6b411          27.4           228566  228566   
15968  content_66b4046cc144          26.6           217415  217415   
15405  content_a023517539fe          85.8           214047  214047   
19332  content_b511d4bc4ad2          27.9           205915  205915   
9200   content_c5063073d048          12.5           192205  192205   
2267   content_f02b48f88241          25.8           181514  181514   

                 reason_code      action  
19636  weak_position_visible  REVIEW_CTR  
29400  weak_position_visible  REVIEW_CTR  
26798  weak_position_visible  REVIEW_CTR  
23767  weak_position_visible  REVIEW_CTR  
26304  weak_po

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [18]:
# ── Build the full ranked queue and write the CSV ──
queue = df_pos.sort_values("score", ascending=False)[
    ["content_id", "score", "reason_code", "action"]
].reset_index(drop=True)

import os
os.makedirs("/workspaces/assignment1/work/outputs", exist_ok=True)
queue.to_csv("/workspaces/assignment1/work/outputs/baseline_action_score.csv", index=False)

print(f"Wrote {len(queue)} rows to baseline_action_score.csv")
print(queue.head(10))

Wrote 28795 rows to baseline_action_score.csv
             content_id   score            reason_code      action
0  content_2cb567c3c89b  497727  weak_position_visible  REVIEW_CTR
1  content_2dba2b1f9536  443434  weak_position_visible  REVIEW_CTR
2  content_b28d1efd668f  286608  weak_position_visible  REVIEW_CTR
3  content_813e88069237  233561  weak_position_visible  REVIEW_CTR
4  content_ff94c9b6b411  228566  weak_position_visible  REVIEW_CTR
5  content_66b4046cc144  217415  weak_position_visible  REVIEW_CTR
6  content_a023517539fe  214047  weak_position_visible  REVIEW_CTR
7  content_b511d4bc4ad2  205915  weak_position_visible  REVIEW_CTR
8  content_c5063073d048  192205  weak_position_visible  REVIEW_CTR
9  content_f02b48f88241  181514  weak_position_visible  REVIEW_CTR


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [19]:
top10 = df_pos.sort_values("score", ascending=False).head(10)
print(top10[["content_id", "avg_position", "impressions_90d", "ctr", "score", "reason_code", "action"]])

                 content_id  avg_position  impressions_90d   ctr   score  \
19636  content_2cb567c3c89b          22.2           497727  0.10  497727   
29400  content_2dba2b1f9536          27.9           443434  0.21  443434   
26798  content_b28d1efd668f          26.2           286608  0.06  286608   
23767  content_813e88069237          26.2           233561  0.06  233561   
26304  content_ff94c9b6b411          27.4           228566  0.04  228566   
15968  content_66b4046cc144          26.6           217415  0.03  217415   
15405  content_a023517539fe          85.8           214047  0.01  214047   
19332  content_b511d4bc4ad2          27.9           205915  0.14  205915   
9200   content_c5063073d048          12.5           192205  0.24  192205   
2267   content_f02b48f88241          25.8           181514  0.10  181514   

                 reason_code      action  
19636  weak_position_visible  REVIEW_CTR  
29400  weak_position_visible  REVIEW_CTR  
26798  weak_position_visible  REVI

1. content_2cb567c3c89b — Action: REVIEW_CTR. Reason: weak_position_visible
   (position 22.2, 497,727 impressions). Confidence: high — huge audience,
   CTR only 0.10%. Wrong if: position 22 is a temporary dip, not a stable rank.

2. content_2dba2b1f9536 — Action: REVIEW_CTR. Reason: weak_position_visible
   (position 27.9, 443,434 impressions). Confidence: medium — CTR 0.21% isn't
   the lowest in the list. Wrong if: query intent is informational, where
   0.21% is actually normal for this position.

3. content_b28d1efd668f — Action: REVIEW_CTR. Reason: weak_position_visible
   (position 26.2, 286,608 impressions). Confidence: high — CTR just 0.06%
   against real volume. Wrong if: title/meta was already rewritten recently
   and this is stale pre-change data.

4. content_813e88069237 — Action: REVIEW_CTR. Reason: weak_position_visible
   (position 26.2, 233,561 impressions). Confidence: medium — near-identical
   profile to #3. Wrong if: #3 and #4 are near-duplicate content
   cannibalizing the same query — fix would be consolidation, not CTR.

5. content_ff94c9b6b411 — Action: REVIEW_CTR. Reason: weak_position_visible
   (position 27.4, 228,566 impressions). Confidence: high — CTR 0.04%, lowest
   relative to volume so far. Wrong if: a stronger competitor is simply
   outranking this page and no snippet change will fix it.

6. content_66b4046cc144 — Action: REVIEW_CTR. Reason: weak_position_visible
   (position 26.6, 217,415 impressions). Confidence: medium — CTR 0.03% is
   the lowest raw number in the top 10. Wrong if: traffic here is mostly
   branded/navigational, where low organic CTR isn't a real snippet problem.

7. content_a023517539fe — Action: REVIEW_CTR. Reason: weak_position_visible
   (position 85.8, 214,047 impressions). Confidence: LOW — position 85 is far
   outside where a CTR/snippet fix helps; flagged mainly by volume. Wrong if:
   this needs a ranking fix, not a CTR fix — likely a weak pick.

8. content_b511d4bc4ad2 — Action: REVIEW_CTR. Reason: weak_position_visible
   (position 27.9, 205,915 impressions). Confidence: medium — CTR 0.14% is
   moderate. Wrong if: 0.14% is already close to expected CTR for position
   ~28 (Signal 2's 11-20 bucket averaged ~0.32%, so this may be near-normal).

9. content_c5063073d048 — Action: REVIEW_CTR. Reason: weak_position_visible
   (position 12.5, 192,205 impressions). Confidence: medium — best position
   in the top 10 but CTR only 0.24%, below the ~0.32-0.65% expected there.
   Wrong if: this page recently dropped from a better rank and CTR just
   hasn't caught up yet.

10. content_f02b48f88241 — Action: REVIEW_CTR. Reason: weak_position_visible
    (position 25.8, 181,514 impressions). Confidence: medium — CTR 0.10%,
    consistent with the pattern. Wrong if: heavy AI-overview/featured-snippet
    presence on this query suppresses CTR regardless of snippet quality.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks:

- content_a023517539fe (#7) is the clearest weak pick. It only made the top 10
  because of impression volume (214,047) — its position, 85.8, is far outside
  where a CTR/snippet fix would realistically help. This is closer to a ranking
  problem than a CTR problem, and probably shouldn't share the same
  weak_position_visible reason code as the others, which cluster around
  position ~22-28.

- content_c5063073d048 (#9) is a softer weak pick: it has the best position in
  the top 10 (12.5) but scored lower on the list, driven mostly by its slightly
  lower impression count. Its CTR (0.24%) is close to what Signal 2 showed as
  the expected range (~0.32%) for that position bucket, so the "problem" here
  may be smaller than the score suggests — worth a lower-priority label if this
  rule gets revised.

Leakage check:

- The rule uses only avg_position and impressions_90d — both are current-state
  descriptive signals, not derived from any outcome label.
- trend_direction and trend_pct were deliberately excluded from both signal
  checks and the rule, since is_declining_label is derived from them
  (per the flyrank-data skill's label trap warning).
- No future-window columns were used — impressions_90d and ctr reflect the
  same trailing 90-day window the whole starter dataset is built on, not a
  forward-looking period.
- content_id and client_id were used only for identification/display, never
  as scoring inputs.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.